# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Roselyn-Koech/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one reporting date.
For this data contract, I will work with the March 2026 mid panel month. The unit of analysis is therefore one client-content-date observation within that month.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature:
• gsc_impressions
• gsc_clicks
• gsc_avg_position
• gsc_data_available
• query_count

Label:
• A future decline in impressions, defined as whether impressions fall by more than 20% in the outcome window.

Context:
• client_hash_id
• content_hash_id
• report_date

Excluded:
• Client names and identifying information, because the analysis should remain pseudonymized.
• The final June 2026 month, because it is the sealed outcome period and could cause leakage when developing the features or label.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [40]:
%pip -q install -U duckdb huggingface_hub

In [41]:
import os
import getpass
import duckdb

print("DuckDB version:", duckdb.__version__)

# Get Hugging Face token from Colab Secret
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

# Connect DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("DuckDB connected successfully.")

DuckDB version: 1.3.2
DuckDB connected successfully.


In [42]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT
            client_hash_id || '|' ||
            content_hash_id || '|' ||
            CAST(report_date AS VARCHAR)
        ) AS unique_client_content_date,
        COUNT(*) -
        COUNT(DISTINCT
            client_hash_id || '|' ||
            content_hash_id || '|' ||
            CAST(report_date AS VARCHAR)
        ) AS duplicate_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_date,duplicate_rows
0,9841378,9841378,0


In [43]:
date_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [44]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS unavailable_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,unavailable_rows
0,9841378,3611061,6230317


Five features

The following five features are used as decision-support signals for identifying content that may need review or refresh:

1. GSC impressions: available at the decision moment because they describe observed search visibility before the outcome window.

2. GSC clicks: available at the decision moment because they describe observed search traffic before the outcome window.

3. GSC average position: available at the decision moment because it describes observed ranking performance before the outcome window.

4. GSC data availability: available at the decision moment because it indicates whether GSC observations are available for the row.

5. Query count: available at the decision moment because it summarizes the number of visible queries associated with the content item from the available 90-day query history.

In [45]:
features_march = con.sql(f"""
    WITH query_counts AS (
        SELECT
            content_hash_id,
            MAX(content_visible_query_count) AS query_count
        FROM {TABLES['fact_query_90d']}
        GROUP BY content_hash_id
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        f.gsc_data_available,
        COALESCE(q.query_count, 0) AS query_count

    FROM {TABLES['fact_daily']} f

    LEFT JOIN query_counts q
        ON f.content_hash_id = q.content_hash_id

    WHERE f.report_date >= DATE '2026-03-01'
      AND f.report_date < DATE '2026-04-01'
      AND f.gsc_data_available IS TRUE

    LIMIT 10000
""").df()

print(f"Feature rows: {len(features_march):,}")
print(f"Unique client-content-date rows: {features_march[['client_hash_id', 'content_hash_id', 'report_date']].drop_duplicates().shape[0]:,}")

features_march.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 10,000
Unique client-content-date rows: 10,000


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,query_count
0,client_08a6a72ff48e62c0,content_5f6fae04728d32ab,2026-03-28,3,0,0.000000,True,5
1,client_08a6a72ff48e62c0,content_5f71205e0b46f70a,2026-03-28,119,1,2.058824,True,12
2,client_08a6a72ff48e62c0,content_5f716493f7989b45,2026-03-24,52,0,4.423077,True,17
3,client_08a6a72ff48e62c0,content_5f8b67a6b0494e15,2026-03-31,7,0,48.142857,True,1
4,client_08a6a72ff48e62c0,content_5f93f846d9834caf,2026-03-28,20,0,0.450000,True,2


In [46]:
# DELIBERATE LEAKAGE EXPERIMENT
# We intentionally use the outcome (future impressions) as a feature.
# This demonstrates why label-derived features must not be used.

leak_data = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-04-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,
        impressions AS current_impressions,
        CASE
            WHEN impressions < 100 THEN 1
            ELSE 0
        END AS is_declining
    FROM monthly
    LIMIT 10000
""").df()

print(f"Leakage experiment rows: {len(leak_data):,}")
leak_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leakage experiment rows: 10,000


,client_hash_id,content_hash_id,current_impressions,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,1
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,0
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,1


In [47]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_leak = leak_data[['current_impressions']]
y_leak = leak_data['is_declining']

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.25,
    random_state=42,
    stratify=y_leak
)

leak_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

leak_model.fit(X_train, y_train)
leak_preds = leak_model.predict(X_test)

print("DELIBERATE LEAKAGE RESULT")
print(classification_report(y_test, leak_preds, digits=3))


DELIBERATE LEAKAGE RESULT
              precision    recall  f1-score   support

           0      1.000     1.000     1.000      2047
           1      1.000     1.000     1.000       453

    accuracy                          1.000      2500
   macro avg      1.000     1.000     1.000      2500
weighted avg      1.000     1.000     1.000      2500



### Leakage lesson

The deliberate leakage experiment produced a perfect 1.000 accuracy because `current_impressions` was used to create the `is_declining` label and then was also given to the model as a feature. This is not a valid predictive result because the feature contains information from the outcome itself.

I therefore remove the label-derived feature and do not use it in the final feature set. The 1.000 score should be treated as a leakage demonstration, not as model performance.

In [48]:
# Remove the deliberately leaked outcome-derived feature.
# These are the five legitimate features kept for decision-support analysis.

honest_feature_cols = [
    'gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'gsc_data_available',
    'query_count'
]

honest_features = features_march[
    ['client_hash_id', 'content_hash_id', 'report_date'] + honest_feature_cols
].copy()

print("Honest feature set:")
print(honest_feature_cols)
print(f"Rows: {len(honest_features):,}")
print("Leaked feature present:", 'current_impressions' in honest_features.columns)

Honest feature set:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available', 'query_count']
Rows: 10,000
Leaked feature present: False


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has several limitations. Client history is unbalanced, so not every client has the same amount of historical data available. Some early rows may have GSC data unavailable, which means search performance features cannot always be calculated consistently. The time windows can also overlap when creating features and outcome periods, so the decision window and future outcome window must be kept separate to avoid leakage.

The March 2026 slice contains 9,841,378 rows, but only 3,611,061 rows have GSC data marked as available. Therefore, conclusions based on GSC performance should be treated as measured signals for the available subset rather than as complete coverage of every row.

In [49]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.